# 11 — ML: Clasificación de Anomalías (Isolation Forest) — Dashboard 9

**Objetivo:** Detectar viajes anómalos (posible fraude tarifario, errores de datos o comportamiento inusual).

**Justificación académica:**
Isolation Forest es un algoritmo de detección de anomalías no supervisado que aísla observaciones
inusuales en pocas particiones. Es ideal para detectar viajes con tarifas sospechosamente altas,
distancias imposibles o combinaciones raras de tarifa + distancia + tiempo.

**Flujo:**
1. Lee `mart_financial_performance` y `mart_operational_profile` desde `tlc_gold`.
2. Construye features por viaje-zona: tarifa/km, propina/km, velocidad, duración.
3. Entrena Isolation Forest.
4. Clasifica zonas/periodos como normales o anómalos.
5. Guarda resultados en `tlc_gold.ml_anomaly_zones`.

In [1]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from src.spark_utils import get_spark, read_mongo, write_mongo
from core.config.settings import settings
import pyspark.sql.functions as F

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import joblib
from pathlib import Path

MODELS_DIR = Path('/home/jovyan/work/data/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR = Path('/home/jovyan/work/data/charts')
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print('Imports OK.')

Imports OK.


In [2]:
spark = get_spark('TLC_ML_Anomaly')
print(f'Spark {spark.version} ready.')

2026-07-19 16:56:17 | INFO     | tlc.spark_utils | [SPARK] Session ready | version=3.4.1 master=local[*]
Spark 3.4.1 ready.


In [3]:
# ── Leer Data Marts desde Gold
df_fin = read_mongo(spark, settings.MONGO_DB_GOLD, 'mart_financial_performance')
df_fin = df_fin.withColumn('tip_to_fare_ratio', F.when(F.col('total_fare') > 0, F.col('total_tips') / F.col('total_fare')).otherwise(0.0))
df_ops = read_mongo(spark, settings.MONGO_DB_GOLD, 'mart_operational_profile')

# ── Construir features para Isolation Forest a nivel zona+mes+hora
# Financiero: tarifa por km, propina por km, ratio propina/tarifa
fin_features = (
    df_fin
    .groupBy('pickup_zone_id', 'zone_name', 'borough', 'vehicle_type', 'year', 'month')
    .agg(
        F.avg('avg_revenue_per_trip').alias('avg_fare'),
        F.avg('avg_tip').alias('avg_tip'),
        F.avg('tip_rate_pct').alias('tip_rate_pct'),
        F.sum('total_trips').alias('total_trips'),
        F.avg('tip_to_fare_ratio').alias('tip_to_fare_ratio'),
    )
)

# Operacional: velocidad, duración y distancia
ops_features = (
    df_ops
    .groupBy('pickup_zone_id', 'vehicle_type', 'year', 'month')
    .agg(
        F.avg('avg_speed_mph').alias('avg_speed_mph'),
        F.avg('avg_duration_min').alias('avg_duration_min'),
        F.avg('avg_distance_miles').alias('avg_distance_miles'),
    )
)

# Join de features
features_spark = (
    fin_features
    .join(ops_features, on=['pickup_zone_id', 'vehicle_type', 'year', 'month'], how='inner')
    # Feature derivado: tarifa por milla (ratio clave para fraude)
    .withColumn(
        'fare_per_mile',
        F.when(F.col('avg_distance_miles') > 0,
               F.col('avg_fare') / F.col('avg_distance_miles'))
        .otherwise(None)
    )
    .withColumn(
        'fare_per_min',
        F.when(F.col('avg_duration_min') > 0,
               F.col('avg_fare') / F.col('avg_duration_min'))
        .otherwise(None)
    )
)

features_pd = features_spark.toPandas()
print(f'Features construidas: {len(features_pd):,} registros (zona × vehículo × mes)')

2026-07-19 16:56:19 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_gold.mart_financial_performance
2026-07-19 16:56:21 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_gold.mart_operational_profile
Features construidas: 34,616 registros (zona × vehículo × mes)


In [4]:
# ── Preparar X para Isolation Forest
ANOMALY_FEATURES = [
    'avg_fare', 'avg_tip', 'tip_rate_pct', 'tip_to_fare_ratio',
    'avg_speed_mph', 'avg_duration_min', 'avg_distance_miles',
    'fare_per_mile', 'fare_per_min',
]

X_raw = features_pd[ANOMALY_FEATURES].fillna(0).replace([np.inf, -np.inf], 0)

# Escalar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f'Matriz de features para Isolation Forest: {X_scaled.shape}')

Matriz de features para Isolation Forest: (34616, 9)


In [5]:
# ── Entrenar Isolation Forest
# contamination=0.05 → esperamos que ~5% de los registros sean anómalos
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1,  # Usar todos los núcleos disponibles
)
iso_forest.fit(X_scaled)

# Predecir: 1 = Normal, -1 = Anomalía
predictions  = iso_forest.predict(X_scaled)
anomaly_score = iso_forest.score_samples(X_scaled)

features_pd['anomaly_prediction'] = predictions
features_pd['anomaly_score']      = anomaly_score
features_pd['is_anomaly']         = features_pd['anomaly_prediction'] == -1
features_pd['anomaly_label']      = features_pd['is_anomaly'].map({True: 'ANOMALÍA', False: 'NORMAL'})

n_anomalies = features_pd['is_anomaly'].sum()
pct_anomaly = n_anomalies / len(features_pd) * 100

print(f'Registros analizados: {len(features_pd):,}')
print(f'Anomalías detectadas: {n_anomalies:,} ({pct_anomaly:.1f}%)')
print(f'Registros normales  : {len(features_pd) - n_anomalies:,}')

# Persistir modelo
joblib.dump(iso_forest, MODELS_DIR / 'isolation_forest_anomaly.pkl', compress=3)
joblib.dump(scaler, MODELS_DIR / 'anomaly_scaler.pkl', compress=3)
print('\n✓ Modelos guardados.')

Registros analizados: 34,616
Anomalías detectadas: 1,731 (5.0%)
Registros normales  : 32,885

✓ Modelos guardados.


In [6]:
# ── Visualización: Top zonas anómalas
top_anomalies = (
    features_pd[features_pd['is_anomaly']]
    .groupby(['zone_name', 'borough'])
    .agg({
        'is_anomaly': 'count',
        'anomaly_score': 'mean',
        'avg_fare': 'mean',
        'fare_per_mile': 'mean',
    })
    .rename(columns={'is_anomaly': 'n_anomalous_months'})
    .sort_values('anomaly_score')
    .head(15)
)

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(
    top_anomalies.index.get_level_values('zone_name') + ' (' + top_anomalies.index.get_level_values('borough') + ')',
    top_anomalies['avg_fare'],
    color='#ef4444', alpha=0.8, edgecolor='white'
)
ax.set_title('Top 15 Zonas con Comportamiento Anómalo\n(Tarifa Promedio por Zona)', fontsize=13, fontweight='bold')
ax.set_xlabel('Tarifa Promedio ($)')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'anomaly_01_top_zones.png'), dpi=120)
plt.close()
print('Gráfica Top Zonas Anómalas guardada.')

Gráfica Top Zonas Anómalas guardada.


In [7]:
# ── Visualización PCA: Normal vs Anomalía
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_.cumsum()[-1] * 100

fig, ax = plt.subplots(figsize=(12, 8))
normal_mask  = ~features_pd['is_anomaly']
anomaly_mask =  features_pd['is_anomaly']

ax.scatter(X_pca[normal_mask, 0],  X_pca[normal_mask, 1],
           c='#3b82f6', alpha=0.4, s=20, label='Normal', edgecolors='none')
ax.scatter(X_pca[anomaly_mask, 0], X_pca[anomaly_mask, 1],
           c='#ef4444', alpha=0.9, s=60, label=f'Anomalía ({n_anomalies})',
           edgecolors='black', linewidth=0.5, marker='X')

ax.set_title(f'Isolation Forest — Detección de Anomalías\nPCA (Varianza explicada: {explained:.1f}%)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('PCA Componente 1')
ax.set_ylabel('PCA Componente 2')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'anomaly_02_pca_scatter.png'), dpi=150)
plt.close()
print('Gráfica PCA guardada.')

Gráfica PCA guardada.


In [8]:
# ── Guardar resultados en MongoDB Gold
result_cols = [
    'pickup_zone_id', 'zone_name', 'borough', 'vehicle_type', 'year', 'month',
    'avg_fare', 'avg_tip', 'tip_rate_pct',
    'avg_speed_mph', 'avg_duration_min', 'avg_distance_miles',
    'fare_per_mile', 'fare_per_min',
    'anomaly_score', 'anomaly_label', 'is_anomaly',
]

result_df = features_pd[result_cols].copy()
result_spark = spark.createDataFrame(result_df)
write_mongo(result_spark, settings.MONGO_DB_GOLD, 'ml_anomaly_zones', mode='overwrite')

print(f'✓ {len(result_df):,} registros guardados en tlc_gold.ml_anomaly_zones')
print(f'  ├── Normales  : {(~result_df["is_anomaly"]).sum():,}')
print(f'  └── Anomalías : {result_df["is_anomaly"].sum():,}')

2026-07-19 17:00:16 | INFO     | tlc.spark_utils | [SPARK] Wrote Unknown rows → MongoDB tlc_gold.ml_anomaly_zones (mode=overwrite)
✓ 34,616 registros guardados en tlc_gold.ml_anomaly_zones
  ├── Normales  : 32,885
  └── Anomalías : 1,731
